# $\color{blue}{\text{Export Statistics}}$
### $\color{blue}{\text{p-values, t-test stats, and average tables for each export}}$

In [1]:
import numpy as np
import pandas as pd
import scipy as sp
from scipy.stats import f_oneway
import random
from scipy.stats import sem
from scipy import stats
import math

# $$\color{Turquoise}{\text{Input Section}}$$

$\color{Blue}{\text{1) Paste the path link to the excel file with the thresholding and/or image info in the following format:}}$

In [2]:
Thresh = pd.read_excel(io="C:/Users/PC/Documents/TEST_FIJI-EXPORTS.xlsx", 
                       sheet_name='Threshold')

$\color{Blue}{\text{2) Paste the path link to the file with the general/countif analysis exports in quotations (the first export file is now the import file)}}$

In [3]:
A = "C:/Users/PC/Documents/TEST_PYTHON-EXPORTS.xlsx"



$\color{Blue}{\text{3) Paste the path link to the file you'd like the stats analysis sent to in quotations (this is now the export file)}}$

In [4]:
export_file = "C:/Users/PC/Documents/TEST_FINAL-PYTHON-EXPORTS.xlsx"



$\color{Blue}{\text{4) Input the antibody names that were in the import label columns, making sure the first label column is the first entry}}$

- $\color{Blue}{\text{prevent possible errors by entering exactly as you did in the general analysis codes (all caps, separated by space)}}$
- $\color{Blue}{\text{For countif analysis, enter only the antibody name from the first column exactly how you did in the previous countif codes}}$

In [5]:
label = input("Label name: ").split()



Label name:  Signal1 Signal2 Signal3


$\color{Blue}{\text{5) Input the column name that you want to sort and analyse data by - this is from the thresholding/image info Excel file}}$

$\color{Blue}{\text{Examples:}}$

- $\color{Blue}{\text{File Number}}$
- $\color{Blue}{\text{Mouse}}$

In [6]:
colsort = input("Column name: ").split()



Column name:  Mouse


$\color{Blue}{\text{6) Input a second column name you want to use for sorting and analysing (aka second grouping level)}}$
$\color{Blue}{\text{(required for countif data)}}$

$\color{Blue}{\text{Examples:}}$

- $\color{Blue}{\text{Genotype}}$
- $\color{Blue}{\text{Sex}}$

$\color{Blue}{\text{if none, enter 'none'}}$

In [7]:
colsort2 = input("Column name: ").split()



Column name:  Genotype


$\color{Blue}{\text{7) Export sheet name(s)}}$

$\color{Blue}{\text{For general analysis:}}$
- $\color{Blue}{\text{Input 'none' if automatic sheet names were used for general analysis exports}}$
- $\color{Blue}{\text{Input all 3 sheet names one at a time if custom sheet names were used}}$

$\color{Blue}{\text{For countif analysis:}}$
- $\color{Blue}{\text{Input 'Countif'}}$

$\color{Blue}{\text{Type and enter 'exit' to stop inputting}}$

In [8]:
sheetname = []

while True:   
    input0 = input("Alternate Sheet Names (or 'exit' to stop): ")    
    if input0.lower() == 'exit':
        break
    sheetname.append(input0)

    print(sheetname)

Alternate Sheet Names (or 'exit' to stop):  none


['none']


Alternate Sheet Names (or 'exit' to stop):  exit


$\color{Blue}{\text{8) Is this the first stats analysis being exported to this file? If so, input 'True' and if not, input 'False'}}$

$\color{Blue}{\text{For countif analysis, always enter True}}$

In [9]:
Export = input("Export Boolean: ").split()



Export Boolean:  True


# $$\color{Aquamarine}{\text{Analysis Code Section}}$$

$\color{Black}{\text{Automatic import of export file and sheet names}}$

In [10]:
label_num = len(label)
all_name = 'all '
space = ' '
new_label = label[0]

countif_setup1 = sheetname[0]
countif_setup2 = new_label+space+countif_setup1

if sheetname[0] != 'none':
    if countif_setup1 == 'Countif':
        Var1 = pd.read_excel(io=A, sheet_name= countif_setup2)
    else:
        Var1 = pd.read_excel(io=A, sheet_name= sheetname[0])
        Var2 = pd.read_excel(io=A, sheet_name= sheetname[1])
        Var3 = pd.read_excel(io=A, sheet_name= sheetname[2])
elif sheetname[0] == 'none':
    if label_num == 1:
        sheetname1 = [all_name+new_label.upper()+' Sums', all_name+new_label.upper()+' Counts', all_name+new_label.upper()+' Medians']
        Var1 = pd.read_excel(io=A, sheet_name= sheetname1[0])
        Var2 = pd.read_excel(io=A, sheet_name= sheetname1[1])
        Var3 = pd.read_excel(io=A, sheet_name= sheetname1[2])
    elif label_num != 1:
        Var1 = pd.read_excel(io=A, sheet_name= label[0].upper()+' Mask Sums')
        Var2 = pd.read_excel(io=A, sheet_name= label[0].upper()+' Mask Counts')
        Var3 = pd.read_excel(io=A, sheet_name= label[0].upper()+' Mask Medians')

$\color{Black}{\text{Automatic import of previously exported stats based on boolean entry in instruction 8}}$

In [11]:
if Export[0] == 'False':
    Imports1 = pd.read_excel(io=export_file, sheet_name='P-Values')
    Imports2 = pd.read_excel(io=export_file, sheet_name='Stats')
    Imports3 = pd.read_excel(io=export_file, sheet_name='Sums Averages')
    Imports4 = pd.read_excel(io=export_file, sheet_name='Counts Averages')
    Imports5 = pd.read_excel(io=export_file, sheet_name='Medians Averages')
elif Export[0] == 'True':
    print('First Export')

First Export


$\color{Black}{\text{Boolean identifiers for dataset configuration}}$

In [12]:
if colsort2[0] == 'none':                                   # one-way ANOVA (one indexing group)
    ID = 1
    print('One Way ANOVA')
elif colsort2[0] != 'none':
    if countif_setup1 == 'Countif':                        # t-tests (two indexing groups for countif data)
        ID = 2
        print('Countif T-test')
    elif countif_setup1 != 'Countif':                      # t-tests (two indexing groups for general analysis)
        ID = 3
        print('General Analysis T-test')


General Analysis T-test


$\color{Black}{\text{Large dataset configuration}}$

In [13]:
step1 = list(Thresh.columns)

if ID == 1:
    Var1a = Var1.drop(step1, axis=1).set_index('index')                          # removes sample identifier columns from results tables
    Var2a = Var2.drop(step1, axis=1).set_index('index')
    Var3a = Var3.drop(step1, axis=1).set_index('index')
    step2 = pd.DataFrame(Thresh[[colsort[0]]]).reset_index()                     # isolates group identifier column from user input
    step2['index'] = step2['index'] + 1
    step3 = step2.set_index('index')                                             # re-indexing new group identifier table
    step4 = pd.concat((step3, Var1a, Var2a, Var3a), axis=1)
    step4[colsort] = step4[colsort].astype(object)                               # combines all results tables and identifier table (non-integer)
    
    step5a = pd.DataFrame(list(step4.columns)).set_axis(['xtra cols'], axis=1)   # ensures no extraneous column from re-indexing remained
    step5b = list(step5a[(step5a['xtra cols'] == 'Unnamed: 0')].count())
    if step5b[0] > 0:
        step5 = step4.drop(['Unnamed: 0'], axis=1)
    else:
        step5 = step4.copy()
elif ID == 2:
    step1a = step1.remove(colsort[0])
    step1b = step1.remove(colsort2[0])
    Var1 = Var1.drop(step1, axis=1)
    step2a = pd.DataFrame(list(Var1.columns)).set_axis(['Cols'], axis=1)
    step2b = step2a[(step2a['Cols'] == colsort2[0])].reset_index()
    step2c = step2a[(step2a['Cols'] == colsort[0])].reset_index()
    step2d = step2b.iloc[0,0]
    step2e = step2c.iloc[0,0]
    colsort1 = colsort+colsort2
    colsort1a = colsort2+colsort
    if step2d > step2e:
        VAR1 = Var1.iloc[0:, step2e:]
    elif step2d < step2e:
        VAR1 = Var1.iloc[0:, step2d:]
    VAR1[colsort] = VAR1[colsort].astype(object)

elif ID == 3:
    step1a = step1.remove(colsort[0])
    step1b = step1.remove(colsort2[0])
    Var1 = Var1.drop(step1, axis=1)
    Var2 = Var2.drop(step1, axis=1)
    Var3 = Var3.drop(step1, axis=1)
    step2a = pd.DataFrame(list(Var1.columns)).set_axis(['Cols'], axis=1)
    step2b = step2a[(step2a['Cols'] == colsort2[0])].reset_index()
    step2c = step2a[(step2a['Cols'] == colsort[0])].reset_index()
    step2d = step2b.iloc[0,0]
    step2e = step2c.iloc[0,0]
    colsort1 = colsort+colsort2
    colsort1a = colsort2+colsort
    if step2d > step2e:
        VAR1 = Var1.iloc[0:, step2e:]
        VAR2 = Var2.iloc[0:, step2e:]
        VAR3 = Var3.iloc[0:, step2e:]
    elif step2d < step2e:
        VAR1 = Var1.iloc[0:, step2d:]
        VAR2 = Var2.iloc[0:, step2d:]
        VAR3 = Var3.iloc[0:, step2d:]
    VAR1[colsort] = VAR1[colsort].astype(object)
    VAR2[colsort] = VAR2[colsort].astype(object)
    VAR3[colsort] = VAR3[colsort].astype(object)
    step4a = VAR1.groupby(colsort1).mean()
    step4b = VAR2.groupby(colsort1).mean()
    step4c = VAR3.groupby(colsort1).mean()
    step5a = step4a.sort_values(by=colsort1a)
    step5b = step4b.sort_values(by=colsort1a)
    step5c = step4c.sort_values(by=colsort1a)
    
else:
    print('error')


$\color{Black}{\text{Calculations from configurated tables}}$

In [14]:
if ID == 1:
    step6 = pd.DataFrame(step5.groupby(colsort).count())                           
    n = step6.iloc[0, 0]                                                         # counts num of values per variable per sample
    means1 = pd.DataFrame(step5.groupby(colsort).mean()).sort_values(by=colsort) # avgs of each sample's values per variable
    newsd = pd.DataFrame(means1.std()).set_axis(['SD'], axis='columns').T        # st dev of all sample values per variable
    newse = pd.DataFrame(means1.sem()).set_axis(['SE'], axis='columns').T        # st error of all sample values per variable
    newvar = pd.DataFrame(means1.var()).set_axis(['Var'], axis='columns').T      # variation of all sample values per variable

    newvar1 = step5.T
    newvar1.columns = newvar1.iloc[0, 0:]
    newvar2 = newvar1.drop(colsort)                                              # table from above flipped, sample identifiers become column names
    newvar2 = newvar2.apply(pd.to_numeric)
    step7 = list(means1.columns)                                                 # list of all variables
    dataframes = [newvar2.iloc[:, i:i+n] for i in range(0, len(newvar2.columns), n)]  # table separated by sample identifier
    step8 = len(list(newvar2.T.columns))                                              # total num of variables
    step9 = step8*n                                                                 # total num of unique variable x sample ID combos

elif ID == 2:
    step4 = VAR1.groupby(colsort1).mean()
    step5 = step4.sort_values(by=colsort1a)
    means1 = pd.DataFrame(step5.groupby(colsort2).mean()).set_axis(['C Mean', 'NLS Mean'], axis=0)
    sd1 = pd.DataFrame(step5.groupby(colsort2).std()).set_axis(['C SD', 'NLS SD'], axis=0)
    se1 = pd.DataFrame(step5.groupby(colsort2).sem()).set_axis(['C SE', 'NLS SE'], axis=0)
    v1 = pd.DataFrame(step5.groupby(colsort2).var()).set_axis(['C Var', 'NLS Var'], axis=0)
    n = pd.DataFrame(step5.groupby(colsort2).count()).set_axis(['C n', 'NLS n'], axis=0)
    df_list = list(n.count())
    df0 = ((n+n)-df_list[0]).set_axis(['C df', 'NLS df'], axis=0)
    df1a = pd.DataFrame((n.sum()-df_list[0])).set_axis(['C df'], axis=1)
    df1b = pd.DataFrame((n.sum()-df_list[0])).set_axis(['NLS df'], axis=1)
    df = pd.concat((df1a, df1b), axis=1, ignore_index=False).T
    cols = list(means1.columns)
    STATS1 = pd.concat((means1, sd1, se1, v1, n, df), axis=0)
    stats1 = STATS1.reset_index().iloc[0:,1:]

    math1a = stats1.iloc[2:4].reset_index().drop('index', axis=1)
    math1b = stats1.iloc[8:10].reset_index().drop('index', axis=1)
    math1c = math1a*math1a
    math1 = (math1b-1)*(math1c)
    math2 = pd.DataFrame(math1.iloc[0]+math1.iloc[1]).T
    math3a = df.reset_index().drop('index', axis=1).iloc[0:1]
    math3 = math2 / math3a
    math4 = pd.DataFrame(stats1.iloc[0] - stats1.iloc[1]).T
    math5 = pd.DataFrame((1 / math1b.iloc[0])+(1 / math1b.iloc[1])).T
    math6 = np.sqrt(math3*math5)

elif ID == 3:
    means1 = pd.DataFrame(step5a.groupby(colsort2).mean()).set_axis(['C Mean', 'NLS Mean'], axis=0)
    means2 = pd.DataFrame(step5b.groupby(colsort2).mean()).set_axis(['C Mean', 'NLS Mean'], axis=0)
    means3 = pd.DataFrame(step5c.groupby(colsort2).mean()).set_axis(['C Mean', 'NLS Mean'], axis=0)
    sd1 = pd.DataFrame(step5a.groupby(colsort2).std()).set_axis(['C SD', 'NLS SD'], axis=0)
    sd2 = pd.DataFrame(step5b.groupby(colsort2).std()).set_axis(['C SD', 'NLS SD'], axis=0)
    sd3 = pd.DataFrame(step5c.groupby(colsort2).std()).set_axis(['C SD', 'NLS SD'], axis=0)
    se1 = pd.DataFrame(step5a.groupby(colsort2).sem()).set_axis(['C SE', 'NLS SE'], axis=0)
    se2 = pd.DataFrame(step5b.groupby(colsort2).sem()).set_axis(['C SE', 'NLS SE'], axis=0)
    se3 = pd.DataFrame(step5c.groupby(colsort2).sem()).set_axis(['C SE', 'NLS SE'], axis=0)
    v1 = pd.DataFrame(step5a.groupby(colsort2).var()).set_axis(['C Var', 'NLS Var'], axis=0)
    v2 = pd.DataFrame(step5b.groupby(colsort2).var()).set_axis(['C Var', 'NLS Var'], axis=0)
    v3 = pd.DataFrame(step5c.groupby(colsort2).var()).set_axis(['C Var', 'NLS Var'], axis=0)
    n = pd.DataFrame(step5a.groupby(colsort2).count()).set_axis(['C n', 'NLS n'], axis=0)
    df_list = list(n.count())
    df0 = ((n+n)-df_list[0]).set_axis(['C df', 'NLS df'], axis=0)
    df1a = pd.DataFrame((n.sum()-df_list[0])).set_axis(['C df'], axis=1)
    df1b = pd.DataFrame((n.sum()-df_list[0])).set_axis(['NLS df'], axis=1)
    df = pd.concat((df1a, df1b), axis=1, ignore_index=False).T
    cols = list(means1.columns)
    STATS1 = pd.concat((means1, sd1, se1, v1, n, df), axis=0)
    STATS2 = pd.concat((means2, sd2, se2, v2, n, df), axis=0)
    STATS3 = pd.concat((means3, sd3, se3, v3, n, df), axis=0)
    stats1 = STATS1.reset_index().iloc[0:,1:]
    stats2 = STATS2.reset_index().iloc[0:,1:]
    stats3 = STATS3.reset_index().iloc[0:,1:]    
    
    math1a = stats1.iloc[2:4].reset_index().drop('index', axis=1)
    math1b = stats1.iloc[8:10].reset_index().drop('index', axis=1)
    math1c = math1a*math1a
    math1 = (math1b-1)*(math1c)
    math2 = pd.DataFrame(math1.iloc[0]+math1.iloc[1]).T
    math3a = df.reset_index().drop('index', axis=1).iloc[0:1]
    math3 = math2 / math3a
    math4 = pd.DataFrame(stats1.iloc[0] - stats1.iloc[1]).T
    math5 = pd.DataFrame((1 / math1b.iloc[0])+(1 / math1b.iloc[1])).T
    math6 = np.sqrt(math3*math5)

    math7a = stats2.iloc[2:4].reset_index().drop('index', axis=1)
    math7b = stats2.iloc[8:10].reset_index().drop('index', axis=1)
    math7c = math7a*math7a
    math7 = (math7b-1)*(math7c)
    math8 = pd.DataFrame(math7.iloc[0]+math7.iloc[1]).T
    math9a = df.reset_index().drop('index', axis=1).iloc[0:1]
    math9 = math8 / math9a
    math10 = pd.DataFrame(stats2.iloc[0] - stats2.iloc[1]).T
    math11 = pd.DataFrame((1 / math7b.iloc[0])+(1 / math7b.iloc[1])).T
    math12 = np.sqrt(math9*math11)

    math13a = stats3.iloc[2:4].reset_index().drop('index', axis=1)
    math13b = stats3.iloc[8:10].reset_index().drop('index', axis=1)
    math13c = math13a*math13a
    math13 = (math13b-1)*(math13c)
    math14 = pd.DataFrame(math13.iloc[0]+math13.iloc[1]).T
    math15a = df.reset_index().drop('index', axis=1).iloc[0:1]
    math15 = math14 / math15a
    math16 = pd.DataFrame(stats3.iloc[0] - stats3.iloc[1]).T
    math17 = pd.DataFrame((1 / math13b.iloc[0])+(1 / math13b.iloc[1])).T
    math18 = np.sqrt(math15*math17)
    
else:
    print('error')

$\color{Black}{\text{Stats tests}}$

In [15]:
if ID == 1:
    dictlist = []
    for i in range(0, step8):                                                 # separation of all unique combos from above
        for j in range(0, n):
            newdict = dataframes[j].iloc[i:i+1]
            dictlist.append(newdict)
    dictlist1 = []
    for i in range(0, step9):                                                 # reconfig and without sample identifier columns
        dictlist2 = dictlist[i].T.reset_index().iloc[0:,1:]
        dictlist1.append(dictlist2)
    order = list(range(0, step9+1, n))                                        # index ranges for each variable
    order1 = len(order)                                                       # number of variables + 1 for loop range
    sort1 = {}
    for i in range(1, order1):                                                # dictionary list separation/key labelling for each variable
        sort1[f'var_{i}'] = dictlist1[order[i-1]:order[i]]
    step9a = step9 / n                                                        # float of total num of variables
    step9b = step9a / 3                                                       # float of total num of unique variables per result data category
    step9c = int(step9b)                                                      # integer of total num of unique variables
    step9d = newvar2.reset_index().reset_index()
    step9e = list(step9d['index'])                                            
    step9f = step9e[0:step9c]                                                 # list of all unique variables
    split_list = [step9f.split(',') for step9f in step9f]                     # separation within list of all unique variables
    fstats = []
    pvals = []
    for key, group_list in sort1.items():                                     # looping through each key within dictionary
        valid_groups = []   
        for group in group_list:
            for i in range(0, step9c):
                if isinstance(group, pd.DataFrame) and step9f[i] in group.columns:
                    valid_groups.append(group[step9f[i]].values)
        if len(valid_groups) >= 2:                                                  # ANOVA if at least 2 groups exist within keys
            f_stat, p_value = f_oneway(*valid_groups)
            fstats.append(f_stat)
            pvals.append(p_value)
        else:
            print(f"Skipping {key}: Not enough valid groups for ANOVA")

elif ID == 2:
    t_stat1 = math4/math6
    tstat1 = pd.to_numeric(pd.Series(list(t_stat1.iloc[0])), downcast='integer', errors='coerce')
    tstat2 = pd.DataFrame(tstat1).T
    pval1 = 2*(stats.t.cdf(-abs(tstat1), math3a.iloc[0,0]))
    pval2 = pd.DataFrame(pval1).T
    pval2.columns = cols
    tstat2.columns = cols

elif ID == 3:
    t_stat1 = math4/math6
    tstat1 = pd.to_numeric(pd.Series(list(t_stat1.iloc[0])), downcast='integer', errors='coerce')
    tstat2 = pd.DataFrame(tstat1).T
    pval1 = 2*(stats.t.cdf(-abs(tstat1), math3a.iloc[0,0]))
    pval2 = pd.DataFrame(pval1).T
    pval2.columns = cols
    tstat2.columns = cols

    t_stat2 = math10/math12
    tstat3 = pd.to_numeric(pd.Series(list(t_stat2.iloc[0])), downcast='integer', errors='coerce')
    tstat4 = pd.DataFrame(tstat3).T
    pval3 = 2*(stats.t.cdf(-abs(tstat3), math9a.iloc[0,0]))
    pval4 = pd.DataFrame(pval3).T
    pval4.columns = cols
    tstat4.columns = cols

    t_stat3 = math16/math18
    tstat5 = pd.to_numeric(pd.Series(list(t_stat3.iloc[0])), downcast='integer', errors='coerce')
    tstat6 = pd.DataFrame(tstat5).T
    pval5 = 2*(stats.t.cdf(-abs(tstat5), math15a.iloc[0,0]))
    pval6 = pd.DataFrame(pval5).T
    pval6.columns = cols
    tstat6.columns = cols

else:
    print('error')

$\color{Black}{\text{Final data tables}}$

In [16]:
if ID == 1:
    sums = pd.DataFrame(['Sums']*step9c)
    counts = pd.DataFrame(['Counts']*step9c)
    medians = pd.DataFrame(['Medians']*step9c)
    analysed = pd.concat((sums, counts, medians), axis=0).reset_index().iloc[0:,1:].set_axis(['Result Analysed'], axis='columns')

    Fstats = pd.DataFrame(fstats).set_axis(['F stats'], axis='columns')
    Pvals = pd.DataFrame(pvals).set_axis(['p values'], axis='columns')
    index = pd.DataFrame(step9e).set_axis(['Variables'], axis='columns')
    finaltable = pd.concat((analysed, index, Fstats, Pvals), axis=1).set_index(['Result Analysed'])
    stats1 = pd.concat((analysed, 
                        newsd.T.reset_index(), 
                        newse.T.reset_index().iloc[0:,1:], 
                        newvar.T.reset_index().iloc[0:,1:]), axis=1).set_index(['Result Analysed']).set_axis(['Variables','SD','SE','Var'], axis='columns')

    stats2a = means1.reset_index().T.reset_index()
    stats2a.columns = stats2a.iloc[0]
    stats2b = stats2a.drop(0).reset_index().iloc[0:, 1:]
    stats2 = pd.concat((analysed, stats2b), axis=1).set_index(['Result Analysed'])

elif ID == 2:
    results1 = pd.concat((pval2, tstat2), axis=0).reset_index()
    results1['index'] = ['Countif p-value', 'Countif t stat']
    results2 = results1.set_index('index')
    table1 = results2.copy()
    stats4 = pd.DataFrame(STATS1).reset_index()
    stats4d = ['Countif Stats']*12
    stats4['Stats'] = stats4d
    table2 = stats4.set_index(['Stats', 'index'])
    finaltable1 = table2.copy()
    finaltable2 = step5a.copy()

elif ID == 3:
    results1 = pd.concat((pval2, tstat2, pval4, tstat4, pval6, tstat6), axis=0).reset_index()
    results1['index'] = ['Sums p-value', 'Sums t stat', 'Counts p-value', 'Counts t stat', 'Medians p-value', 'Medians t stat']

    if Export[0] == 'False':
        results2 = results1.set_index('index')
        imports1 = Imports1.set_index('index')
        table1 = pd.concat((imports1, results2), axis=1)
    elif Export[0] == 'True':
        results2 = results1.set_index('index')
        table1 = results2.copy()

    stats4 = pd.concat((STATS1, STATS2, STATS3), axis=0).reset_index()
    stats4a = ['Sum Stats']*12
    stats4b = ['Count Stats']*12
    stats4c = ['Median Stats']*12
    stats4d = stats4a+stats4b+stats4c
    stats4['Stats'] = stats4a+stats4b+stats4c
    table2 = stats4.set_index(['Stats', 'index'])

    if Export[0] == 'False':
        imports2 = Imports2.fillna('')
        new4a = step4a.reset_index()
        new4b = step4b.reset_index()
        new4c = step4c.reset_index()
        finaltable1 = pd.concat((imports2, table2.reset_index()), axis=1, ignore_index=False).iloc[0:,2:].set_index(['Stats', 'index'])
        if colsort2[0] != 'none':
            finaltable2 = pd.concat((new4a, Imports3), axis=1).iloc[0:,2:].set_index(colsort1)
            finaltable3 = pd.concat((new4b, Imports4), axis=1).iloc[0:,2:].set_index(colsort1)
            finaltable4 = pd.concat((new4c, Imports5), axis=1).iloc[0:,2:].set_index(colsort1)
        elif colsort2[0] == 'none':
            finaltable2 = pd.concat((new4a, Imports3), axis=1).iloc[0:,2:].set_index(colsort)
            finaltable3 = pd.concat((new4b, Imports4), axis=1).iloc[0:,2:].set_index(colsort)
            finaltable4 = pd.concat((new4c, Imports5), axis=1).iloc[0:,2:].set_index(colsort)
    elif Export[0] == 'True':
        finaltable1 = table2.copy()
        finaltable2 = step4a.copy()
        finaltable3 = step4b.copy()
        finaltable4 = step4c.copy()

else:
    print('error')

In [17]:
if ID > 1:
    def color(val):
        if val < 0.005:
            return 'background-color: turquoise'
        elif 0.01 > val > 0.005:
            return 'background-color: aquamarine'
        elif 0.05 > val > 0.01:
            return 'background-color: skyblue'
        elif 0.11 > val > 0.05:
            return 'background-color: lightyellow'
        else:
            return 'background-color: white'

    def apply_color_every_other_row(table1):
        styles = pd.DataFrame('', index=table1.index, columns=table1.columns)
        for i, idx in enumerate(table1.index):
            if i % 2 == 0:  
                styles.loc[idx] = table1.loc[idx].apply(color)            
        return styles

    finaltable = table1.style.apply(lambda x: apply_color_every_other_row(table1), axis=None)
else:
    print('N/A')

# $$\color{blue}{\text{Export Section}}$$

In [18]:
if ID == 1:
    with pd.ExcelWriter(export_file, mode='a', if_sheet_exists='replace') as writer:  
        finaltable.to_excel(writer, sheet_name='1Way ANOVA')
        stats1.to_excel(writer, sheet_name='Group Stats')
        stats2.to_excel(writer, sheet_name='Averages')

elif ID == 2:
    with pd.ExcelWriter(export_file, mode='a', if_sheet_exists='replace') as writer:  
        finaltable.to_excel(writer, sheet_name='Countif P-Values')
        finaltable1.to_excel(writer, sheet_name='Countif Stats')
        finaltable2.to_excel(writer, sheet_name='Countif Averages')

elif ID == 3:
    with pd.ExcelWriter(export_file, mode='a', if_sheet_exists='replace') as writer:  
        finaltable.to_excel(writer, sheet_name='P-Values')
        finaltable1.to_excel(writer, sheet_name='Stats')
        finaltable2.to_excel(writer, sheet_name='Sums Averages')
        finaltable3.to_excel(writer, sheet_name='Counts Averages')
        finaltable4.to_excel(writer, sheet_name='Medians Averages')

In [19]:
finaltable

,all SIGNAL1,SIGNAL1 & SIGNAL2,SIGNAL1 no SIGNAL2,SIGNAL1 & SIGNAL2 & SIGNAL3,SIGNAL1 & SIGNAL2 no SIGNAL3,SIGNAL1 & SIGNAL3 & SIGNAL2,SIGNAL1 no SIGNAL2 no SIGNAL3,SIGNAL1 & SIGNAL3,SIGNAL1 no SIGNAL3
index,,,,,,,,,
Sums p-value,0.849990,0.693409,0.609177,0.906583,0.484745,0.420364,0.794728,0.551276,0.897082
Sums t stat,0.194092,-0.405838,0.527759,0.120358,-0.725554,0.840325,0.267222,0.616601,-0.132675
Counts p-value,0.664741,0.697131,0.455911,0.945829,0.600859,0.414605,0.520118,0.589166,0.740911
Counts t stat,0.446512,-0.400613,0.775643,-0.069671,-0.540234,0.851145,0.666588,0.557926,0.339971
Medians p-value,0.052816,0.332298,0.094632,0.447200,0.127610,0.580758,0.024393,0.612164,0.019445
Medians t stat,-2.195793,-1.018834,-1.846203,0.791176,-1.661430,-0.570764,-2.648088,-0.523299,-2.780156
